# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset via Croissant
dataset = mlc.Dataset(croissant_url)

# Show documentation from metadata
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")
print(f"Date published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets and field `@id`s, using the metadata object.

In [ ]:
# List all record sets and their @id
record_sets = dataset.metadata.recordSet

if not record_sets:
    print('No record sets found in the Croissant metadata.')
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
        # List fields for each record set
        if 'field' in rs:
            print('  Fields:')
            for fld in rs['field']:
                print(f"    Field @id: {fld['@id']}, name: {fld.get('name', 'N/A')} (type: {fld.get('dataType', 'unknown')})")
        else:
            print('  No fields defined.')

As an example, let's enumerate the available records and preview their fields using their `@id`.
If you know the `@id` of a record set from above (for example, `'cr:recordSet:proteins'` if it exists), you may use it below.

In [ ]:
# Helper: Try to find likely main record set @id (commonly something like 'cr:recordSet:proteins')
rs_ids = []
if record_sets:
    for rs in record_sets:
        rs_ids.append(rs['@id'])
# For demonstration, pick the first if available
if rs_ids:
    main_record_set_id = rs_ids[0]
    print(f"Using record set @id: {main_record_set_id}")
    # Print sample records
    for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
        print(f"Record {i}: {rec}")
        if i >= 2:
            break
else:
    print('No available record sets to preview records.')

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for further analysis.

In [ ]:
dataframes = {}

if rs_ids:
    for rs_id in rs_ids:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {df.shape[0]} records into dataframe for record set '{rs_id}'.")
            print(f"Columns: {df.columns.tolist()}")
            display(df.head(3))
        else:
            print(f"No records available for record set {rs_id}")
else:
    print('No record sets available. Please check the metadata.')

For the remainder of this notebook, we'll focus on the **main record set** (the first one detected above) for exploratory data analysis.

## 4. Exploratory Data Analysis (EDA)
Let's look for a numeric field for filtering and normalization.

In [ ]:
# Choose main record set DataFrame
if rs_ids and rs_ids[0] in dataframes:
    main_rs = rs_ids[0]
    df = dataframes[main_rs]
    print(f"Analyzing record set '{main_rs}' with shape {df.shape}")
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    print(f"Numeric columns: {numeric_cols}")
    # Try to use the first numeric column or set a default
    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean()  # Example cutoff: mean
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (top 5 rows):")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} (mean ~0, top 5 rows):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Select a group/categoric field
        group_fields = df.select_dtypes(include='object').columns.tolist()
        # Skip over 'description' or 'accession' and pick another, but default to first
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            print(f"Grouped mean {numeric_field} by {group_field} (top 5):")
            display(grouped_df.head())
    else:
        print('No numeric columns found to analyze.')
else:
    print('Main dataframe for analysis unavailable.')

## 5. Visualization
Show distributions and relationships for key fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if rs_ids and rs_ids[0] in dataframes and numeric_cols:
    # Histogram of the main numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if len(numeric_cols) > 1:
        # Scatter plot vs. another numeric field, if available
        numeric_field2 = numeric_cols[1]
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=df[numeric_field], y=df[numeric_field2])
        plt.xlabel(numeric_field)
        plt.ylabel(numeric_field2)
        plt.title(f"{numeric_field} vs {numeric_field2}")
        plt.show()
else:
    print('Insufficient numeric fields or data for visualization.')

## 6. Conclusion
In this notebook, we loaded, explored, and visualized the FAIR² mass spectrometry dataset using `mlcroissant` and pandas. This process included:
- Extracting the available record sets and fields by their `@id`
- Loading data for each record set into DataFrames
- Filtering, normalizing, and grouping quantitative values
- Visualizing data distributions and relationships

For further work, adapt this workflow to focus specifically on biological or clinical analyses relevant to the dataset context. The workflow shown here can be repeated or extended for any dataset with a valid Croissant JSON-LD schema.